# MODFLOW 6: simulación de flujo v 2.0


## Resumen.
Se realiza la misma simulación de flujo realizada en la notebook [03_MF6_GWF_v1.ipynb](03_MF6_GWF_v1.ipynb) pero ahora hacemos uso de las funciones `xmf6.gwf.init_sim()` y `xmf6.gwf.set_packages()` las cuales son más generales y encapsulan la inicialización de una simulación de flujo.

<p xmlns:cc="http://creativecommons.org/ns#" xmlns:dct="http://purl.org/dc/terms/"><a property="dct:title" rel="cc:attributionURL" href="https://github.com/luiggix/xmf6/tree/main/examples/01_mf6_tutorial/">MODFLOW 6: tutorial</a> (03_MF6_GWF_v1.ipynb) by <b>Luis M. de la Cruz Salas (2025)</b> is licensed under <a href="http://creativecommons.org/licenses/by-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">Attribution-ShareAlike 4.0 International<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a>.</p> 

## Paso 0. Importación de bibliotecas y módulos.

Debemos incluir todas las bibliotecas que se van a usar.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import flopy
import xmf6

## Paso 1. Parámetros de la simulación. 

Las funciones `xmf6.gwf.init_sim()` y `xmf6.gwf.set_packages()` reciben como parámetros varios diccionarios en los que va almacenada toda la información requerida para inicializar los objetos de la simulación. En la celda siguiente definimos esta información.

In [2]:
# --- Componentes ---

# Parámetros de la simulación (flopy.mf6.MFSimulation)
init = {
    'sim_name' : "flow",
    'exe_name' : "C:\\Users\\luiggi\\Documents\\GitSites\\xmf6\\mf6\\windows\\mf6",
#    'exe_name' : "../../mf6/macosarm/mf6",
    'sim_ws' : "sandbox4"
}

# Parámetros para el tiempo (flopy.mf6.ModflowTdis)
tdis = {
    'units': "DAYS",
    'nper' : 1,
    'perioddata': [(1.0, 1, 1.0)]
}

# Parámetros para la solución numérica (flopy.mf6.ModflowIms)
ims = {}

# Parámetros para el modelo de flujo (flopy.mf6.ModflowGwf)
gwf = { 
    'modelname': init["sim_name"],
    'model_nam_file': f"{init["sim_name"]}.nam",
    'save_flows': True
}

# --- Paquetes del modelo de flujo ---

# Parámetros para la discretización espacial (flopy.mf6.ModflowGwfdis)
dis = {
    'nlay': 1, 
    'nrow': 20, 
    'ncol': 30,
    'delr': 0.5, 
    'delc': 0.5, 
    'top' : 0.0, 
    'botm': -1.0 
}

# Parámetros para las condiciones iniciales (flopy.mf6.ModflowGwfic)
ic = {
    'strt': 10
}

# Parámetros para las condiciones de frontera (flopy.mf6.ModflowGwfchd)
chd_data = []
for row in range(dis['nrow']):
    chd_data.append([(0, row, 0), 10.0])       # Condición en la pared izquierda
    chd_data.append([(0, row, dis['ncol'] - 1), 5.0]) # Condición en la pared derecha

chd = {
    'stress_period_data': chd_data,     
}

# Parámetros para las propiedades de flujo (flopy.mf6.ModflowGwfnpf)
k_data = np.random.rand(dis['nlay'], dis['nrow'], dis['ncol'])*1.0 
k_data[:,5:15,10:20] = 0.1

npf = {
    'save_specific_discharge': True,
    'k': k_data, 
}

# Parámetros para almacenar y mostrar la salida de la simulación (flopy.mf6.ModflowGwfoc)
oc = {
    'budget_filerecord': f"{init['sim_name']}.bud",
    'head_filerecord': f"{init['sim_name']}.hds",
    'saverecord': [("HEAD", "ALL"), ("BUDGET", "ALL")],
    'printrecord': [("HEAD", "ALL")]
}

Todos los diccionarios definidos antes se pueden revisar usando la función `xmf6.nice_print()`:

In [3]:
xmf6.nice_print(init, "Inicialización de la simulación")
xmf6.nice_print(tdis, "Discretización temporal")
xmf6.nice_print(ims, "Solver")
xmf6.nice_print(gwf, "Modelo de flujo")
xmf6.nice_print(dis, "Discretización espacial")
xmf6.nice_print(ic, "Condiciones iniciales")
xmf6.nice_print(chd, "Condiciones de frontera")
xmf6.nice_print(npf, "Parámetros de flujo")
xmf6.nice_print(oc, "Salida de la simulación")


Inicialización de la simulación
―――――――――――――――――――――――――――――――
sim_name = flow
exe_name = C:\Users\luiggi\Documents\GitSites\xmf6\mf6\windows\mf6
  sim_ws = sandbox4
―――――――――――――――――――――――――――――――

Discretización temporal
―――――――――――――――――――――――
     units = DAYS
      nper = 1
perioddata = ―― data array ――
           = (1.0, 1, 1.0)
―――――――――――――――――――――――

Solver
――――――
――――――

Modelo de flujo
―――――――――――――――
     modelname = flow
model_nam_file = flow.nam
    save_flows = True
―――――――――――――――

Discretización espacial
―――――――――――――――――――――――
nlay = 1
nrow = 20
ncol = 30
delr = 0.5
delc = 0.5
 top = 0.0
botm = -1.0
―――――――――――――――――――――――

Condiciones iniciales
―――――――――――――――――――――
strt = 10
―――――――――――――――――――――

Condiciones de frontera
―――――――――――――――――――――――
stress_period_data = ―― data array ――
                   = [(0, 0, 0), 10.0]
                   = [(0, 0, 29), 5.0]
                   = [(0, 1, 0), 10.0]
                   = [(0, 1, 29), 5.0]
                   = [(0, 2, 

## Paso 2. Componentes.

La función `xmf6.gwf.init_sim()` inicializa la simulación creando la siguiente estructura:

<figure>
  <img src="../figures/MF6_componentes.png" width=250px>
  <figcaption>Figura 1. Tomada de [2].</figcaption>
</figure> 

Esta función regresa un objeto de tipo `flopy.mf6.modflow.mfsimulation.MFSimulation` para manejar la simulación. En la celda siguiente ejecutamos esta función:

In [4]:
o_sim = xmf6.gwf.init_sim(init = init, tdis = tdis, ims = ims, silent = False)



sim configuration
――――――――――――――――――
sim_name = flow
exe_name = C:\Users\luiggi\Documents\GitSites\xmf6\mf6\windows\mf6
  sim_ws = sandbox4
――――――――――――――――――


time configuration
―――――――――――――――――――
     units = DAYS
      nper = 1
perioddata = ―― data array ――
           = (1.0, 1, 1.0)
―――――――――――――――――――


numerical solution configuration
―――――――――――――――――――――――――――――――――
―――――――――――――――――――――――――――――――――


In [5]:
print(type(o_sim))
print(o_sim)

<class 'flopy.mf6.modflow.mfsimulation.MFSimulation'>
sim_name = flow
sim_path = C:\Users\luiggi\Documents\GitSites\xmf6\examples\00_mf6_tutorial\sandbox4
exe_name = C:\Users\luiggi\Documents\GitSites\xmf6\mf6\windows\mf6

###################
Package mfsim.nam
###################

package_name = mfsim.nam
filename = mfsim.nam
package_type = nam
model_or_simulation_package = simulation
simulation_name = flow


###################
Package flow.tdis
###################

package_name = flow.tdis
filename = flow.tdis
package_type = tdis
model_or_simulation_package = simulation
simulation_name = flow


###################
Package ims_-1
###################

package_name = ims_-1
filename = flow.ims
package_type = ims
model_or_simulation_package = simulation
simulation_name = flow





## Paso 3. Paquetes.

Ahora creamos el objeto para el modelo de flujo y le agregamos los paquetes requeridos para la simulación. Para ello usamos la función `xmf6.gwf.set_packages()`. Esta función recibe un número variable de parámetros, todos ellos son diccionarios con información para inicializar los paquetes necesarios para la simulación. En este caso la ejecución de función es como sigue:

In [6]:
o_gwf, packages = xmf6.gwf.set_packages(o_sim, silent = True, 
                                        gwf = gwf, 
                                        dis = dis, ic = ic, chd = chd, npf = npf, oc = oc)

La función regresa:
* Un objeto (`o_gwf`) del modelo de flujo de tipo `flopy.mf6.modflow.mfgwf.ModflowGwf`.
* Un diccionario (`packages`) con los objetos de los paquetes agregados al modelo.

In [7]:
print(type(o_gwf))

<class 'flopy.mf6.modflow.mfgwf.ModflowGwf'>


In [8]:
packages.keys()

dict_keys(['dis', 'ic', 'chd', 'npf', 'oc'])

In [9]:
for k, v in packages.items():
    print(v)

package_name = dis
filename = flow.dis
package_type = dis
model_or_simulation_package = model
model_name = flow

Block dimensions
--------------------
nlay
{internal}
(1)

nrow
{internal}
(20)

ncol
{internal}
(30)


Block griddata
--------------------
delr
{constant 0.5}

delc
{constant 0.5}

top
{constant 0.0}

botm
{constant -1.0}



package_name = ic
filename = flow.ic
package_type = ic
model_or_simulation_package = model
model_name = flow

Block griddata
--------------------
strt
{constant 10}



package_name = chd_0
filename = flow.chd
package_type = chd
model_or_simulation_package = model
model_name = flow

Block period
--------------------
stress_period_data
{internal}
(    cellid_layer  cellid_row  cellid_column  head
0              0           0              0  10.0
1              0           0             29   5.0
2              0           1              0  10.0
3              0           1             29   5.0
4              0           2              0  10.0
5            

Flopy contiene también funciones para obtener la lista de paquetes y su contenido

In [10]:
o_gwf.get_package_list()

['DIS', 'IC', 'CHD_0', 'NPF', 'OC']

In [11]:
o_gwf.get_package('DIS')

package_name = dis
filename = flow.dis
package_type = dis
model_or_simulation_package = model
model_name = flow

Block dimensions
--------------------
nlay
{internal}
(1)

nrow
{internal}
(20)

ncol
{internal}
(30)


Block griddata
--------------------
delr
{constant 0.5}

delc
{constant 0.5}

top
{constant 0.0}

botm
{constant -1.0}



## Paso 4. Escritura de archivos iniciales.

In [12]:
o_sim.write_simulation(silent = True)

## Paso 5. Ejecución de la simulación.

In [13]:
o_sim.run_simulation(silent=False)

FloPy is using the following executable to run the model: ..\..\..\mf6\windows\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.6.1 02/10/2025

   MODFLOW 6 compiled Feb 10 2025 17:37:25 with Intel(R) Fortran Intel(R) 64
   Compiler Classic for applications running on Intel(R) 64, Version 2021.7.0
                             Build 20220726_000000

This software has been approved for release by the U.S. Geological 
Survey (USGS). Although the software has been subjected to rigorous 
review, the USGS reserves the right to update the software as needed 
pursuant to further analysis and review. No warranty, expressed or 
implied, is made by the USGS or the U.S. Government as to the 
functionality of the software and related material nor shall the 
fact of release constitute any such warranty. Furthermore, the 
software is released on condition that neither the USGS nor the U.S. 
Gover

(True, [])

## Paso 6. Recuperación de los resultados.

La biblioteca `xmf6` contiene dos funciones para recuperar la información de manera directa:
* `xmf6.gwf.get_head()` : Obtiene un arreglo de numpy con la información de la carga hidráulica.
* `xmf6.gwf.get_specific_discharge()` : Obtiene arreglos de numpy con la información de la descarga específica.

In [ ]:
# --- Recuperamos los resultados de la simulación ---
head = xmf6.gwf.get_head(o_gwf)
qx, qy, qz, n_q = xmf6.gwf.get_specific_discharge(o_gwf, text="DATA-SPDIS")

In [ ]:
# --- Parámetros para las gráficas ---
grid = o_gwf.modelgrid
x, y, z = grid.xyzcellcenters
xticks = np.linspace(grid.extent[0], grid.extent[1], 7)
yticks = np.linspace(grid.extent[2], grid.extent[3], 5)
xlabels = [f'{x:1.1f}' for x in xticks]
ylabels = [f'{y:1.1f}' for y in yticks]
kvmin = 1.0 #np.nanmin(k_data)
kvmax = 0.0 #np.nanmax(k_data)
hvmin = np.nanmin(head)
hvmax = np.nanmax(head)
qvmin = 0.00 #np.nanmin(n_q)
qvmax = 0.35 #np.nanmax(n_q)

# --- Definición de la figura ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize =(10,10))

# --- Gráfica 1. ---
kview = flopy.plot.PlotMapView(model = o_gwf, ax = ax1)
kview.plot_grid(linewidths = 0.5, alpha = 0.5)
k_ac = kview.plot_array(k_data, cmap = "gist_heat", vmin = kvmin, vmax = kvmax)
k_cb = plt.colorbar(k_ac, ax = ax1, label = "$k$", 
                    ticks = [0.0, 0.25, 0.50, 0.75, 1.0],
                    cax = xmf6.vis.cax(ax1, k_ac))
k_cb.ax.tick_params(labelsize=8)
ax1.set_title("Permeabilidad hidráulica $k$", fontsize=10)
ax1.set_ylabel("$y$ (m)", fontsize = 8)
ax1.set_xticks(ticks = xticks, labels = xlabels, fontsize = 8)
ax1.set_yticks(ticks = yticks, labels = ylabels, fontsize = 8)
ax1.set_aspect('equal')

# --- Gráfica 2. ---
hview = flopy.plot.PlotMapView(model = o_gwf, ax = ax2)
h_ac = hview.plot_array(head, cmap = "YlGnBu", vmin = hvmin, vmax = hvmax, alpha = 0.75)
hview.contour_array(head, levels = 30, cmap = "bone", linewidths = 1.0)
ax2.quiver(x, y, qx[0], qy[0], scale = 3, 
           color = 'k', linewidth = 0.95, pivot = 'middle')
h_cb = plt.colorbar(h_ac, ax = ax2, label = "$h$ (m)", 
                    cax = xmf6.vis.cax(ax2, h_ac))
h_cb.ax.tick_params(labelsize=8)
ax2.set_title("Carga hidráulica $h$", fontsize=10)
ax2.set_ylabel("$y$ (m)", fontsize = 8)
ax2.set_xticks(ticks = xticks, labels = xlabels, fontsize = 8)
ax2.set_yticks(ticks = yticks, labels = ylabels, fontsize = 8)
ax2.set_aspect('equal')

# --- Gráfica 3. ---
fview = flopy.plot.PlotMapView(model = o_gwf, ax = ax3)
q_ac = fview.plot_array(n_q, cmap = "winter", vmin = qvmin, vmax = qvmax, alpha = 0.25)
fview.contour_array(head, levels = 20, cmap = 'bone', linewidths = 0.75, )
ax3.streamplot(x, y[::-1][:], qx[0], qy[0][::-1], 
               density = [2, 1.5], linewidth = 0.75, broken_streamlines = True, 
               color = n_q, cmap = "winter", 
               arrowstyle = "->", arrowsize = 0.75,  )
q_cb = plt.colorbar(q_ac, ax=ax3, label="$q$", 
                    ticks = np.linspace(0.0, 0.35, 7),
                    format = "{x:3.2f}",
                    cax = xmf6.vis.cax(ax3, q_ac))
q_cb.ax.tick_params(labelsize=8)
ax3.set_title("Descarga específica $q$", fontsize=10)
ax3.set_xlabel("$x$ (m)", fontsize = 8)
ax3.set_ylabel("$y$ (m)", fontsize = 8)
ax3.set_xticks(ticks = xticks, labels = xlabels, fontsize = 8)
ax3.set_yticks(ticks = yticks, labels = ylabels, fontsize = 8)
ax3.set_aspect('equal')

plt.tight_layout()
plt.savefig("04_MF6.pdf")
plt.show()

# Referencias

[1] Langevin, C. D., Hughes, J. D., Provost, A. M., Russcher, M. J., & Panday, S. (2023). MODFLOW as a configurable Multi‐Model Hydrologic Simulator. Ground Water. https://doi.org/10.1111/gwat.13351.

[2] Langevin, C.D., Hughes, J.D., Provost, A.M., Banta, E.R., Niswonger, R.G., and Panday, Sorab, 2017, Documentation for the MODFLOW 6 Groundwater Flow (GWF) Model: U.S. Geological Survey Techniques and Methods, book 6, chap. A55, 197 p., accessed August 4, 2017. https://doi.org/10.3133/tm6A55.